In [1]:
from rdflib import Graph, URIRef, Namespace, Literal, XSD, RDF
from rdflib.plugins.stores.sparqlstore import SPARQLStore
from tqdm.autonotebook import tqdm
import pandas as pd
from copy import deepcopy
import numpy
from sklearn.metrics.pairwise import cosine_similarity
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import defaultdict
from rapidfuzz import process, fuzz, distance
import math

import graph_similarity

import random

import torch
from transformers import BertTokenizer
from transformers import BertModel
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics.pairwise import cosine_similarity

import concurrent.futures

from rdflib import URIRef, Literal
import string
import dateutil.parser as dparser
import datetime
from rapidfuzz import fuzz, distance
import math
import re
import os

import pickle

C:\Users\Ch.Raghava\AppData\Local\Temp\ipykernel_22500\2187368035.py:3: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
GRAPH_1 = "D:/FTM/Full-Triple-Matcher/dataset/entity-matching/DW-NB/kg1.ttl"

GRAPH_1_INV_FUNC_PATH = "D:/FTM/Full-Triple-Matcher/dataset/entity-matching/DW-NB/kg1_inversability.csv"

GRAPH_2 = "D:/FTM/Full-Triple-Matcher/dataset/entity-matching/DW-NB/kg2.ttl"

GRAPH_2_INV_FUNC_PATH = "D:/FTM/Full-Triple-Matcher/dataset/entity-matching/DW-NB/kg2_inversability.csv"

SUBJECT = 'subject'
PREDICATE = 'predicate'
INVERSE_FUNCTIONALITY = 'inverse_functionality'
INVERSABILITY = 'inverse_functionality'
FUNCTIONALITY = 'functionality'
MAX_LENGTH_FULL_MATCH = 10000
MAX_SAVED_PREDICATE_COUNTER = 10000000
LABEL_PREDICATE = "http://www.w3.org/2000/01/rdf-schema#label"
FILE_FOLDER = "D:/FTM/Full-Triple-Matcher/results/full-triple-matcher/dw-nb//"
GRAPH_1_ABSTRACT = "http://dbkwik.webdatacommons.org/ontology/abstract"
GRAPH_2_ABSTRACT = "http://dbkwik.webdatacommons.org/ontology/abstract"

In [3]:
graph_1_inv_func_df = pd.read_csv(GRAPH_1_INV_FUNC_PATH)

#graph_1_predicate_list = threshold_graph_1_inv_func_df[PREDICATE].tolist()

graph_1 = Graph()
graph_1.parse(GRAPH_1)

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#gYear, Converter=<function parse_date at 0x000002868FE7A0D0>
Traceback (most recent call last):
  File "C:\Users\Ch.Raghava\miniconda3\envs\full-triple-matcher\lib\site-packages\rdflib\term.py", line 2119, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "C:\Users\Ch.Raghava\miniconda3\envs\full-triple-matcher\lib\site-packages\isodate\isodates.py", line 203, in parse_date
    raise ISO8601Error('Unrecognised ISO 8601 date format: %r' % datestring)
isodate.isoerror.ISO8601Error: Unrecognised ISO 8601 date format: '-0609'
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#gYear, Converter=<function parse_date at 0x000002868FE7A0D0>
Traceback (most recent call last):
  File "C:\Users\Ch.Raghava\miniconda3\envs\full-triple-matcher\lib\site-packages\rdflib\term.py", line 2119, in _castLexicalToPython
    return conv_func(le

<Graph identifier=N339e8eba7d2c4c99b1114ccf55bd4bc8 (<class 'rdflib.graph.Graph'>)>

In [4]:
graph_2_inv_func_df = pd.read_csv(GRAPH_2_INV_FUNC_PATH)

#graph_2_predicate_list = threshold_graph_2_inv_func_df[PREDICATE].tolist()

graph_2 = Graph()
graph_2.parse(GRAPH_2)
# graph_2 = Graph("SPARQLStore")
# graph_2.open(GRAPH_2)

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#gYear, Converter=<function parse_date at 0x000002868FE7A0D0>
Traceback (most recent call last):
  File "C:\Users\Ch.Raghava\miniconda3\envs\full-triple-matcher\lib\site-packages\rdflib\term.py", line 2119, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "C:\Users\Ch.Raghava\miniconda3\envs\full-triple-matcher\lib\site-packages\isodate\isodates.py", line 203, in parse_date
    raise ISO8601Error('Unrecognised ISO 8601 date format: %r' % datestring)
isodate.isoerror.ISO8601Error: Unrecognised ISO 8601 date format: '-0546'
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#gYear, Converter=<function parse_date at 0x000002868FE7A0D0>
Traceback (most recent call last):
  File "C:\Users\Ch.Raghava\miniconda3\envs\full-triple-matcher\lib\site-packages\rdflib\term.py", line 2119, in _castLexicalToPython
    return conv_func(le

<Graph identifier=Nc3152d484ad448a182502fe180b35fc3 (<class 'rdflib.graph.Graph'>)>

In [5]:
def get_last_part_url(uri):
    return uri.rsplit('/', 1)[1]

In [6]:
def get_label_graph(graph, predicate_df, uri_column):
    predicate_label_list = list()
    LABEL = "http://www.w3.org/2000/01/rdf-schema#label"
    ALT_LABEL = "http://www.w3.org/2004/02/skos/core#altLabel"
    label_predicate = URIRef(LABEL)
    alt_label_predicate = URIRef(ALT_LABEL)
    for index, row in tqdm(predicate_df.iterrows(), total=len(predicate_df)):
        search_uriref = URIRef(row[uri_column])
        label_list = list()
        for label in graph.objects(search_uriref, label_predicate):
            label_list.append(str(label))
        for label in graph.objects(search_uriref, alt_label_predicate):
            label_list.append(str(label))
        if len(label_list) == 0:
            label_list.append(get_last_part_url(row[uri_column]))
        predicate_label_list.append(label_list)
    return predicate_label_list

In [7]:
def calculate_similarity_using_label(df_1, df_2, uri_column):
    label_list_1 = get_label_graph(graph_1, df_1, uri_column)
    df_1['label'] = label_list_1
    
    label_list_2 = get_label_graph(graph_2, df_2, uri_column)
    df_2['label'] = label_list_2
    
    uri_set_1 = set(df_1[uri_column])
    uri_set_2 = set(df_2[uri_column])
    
    prob_dict = dict()
    
    for uri_1 in uri_set_1:
        if uri_1 in uri_set_2:
            prob_dict.setdefault(uri_1, {})
            prob_dict[uri_1][uri_1] = 1.0
    
    return prob_dict

In [8]:
def match_labels(prob_dict, df_1, df_2, label_column, max_score, uri_column):
    unique_labels = set()
    for label_list in df_1[label_column]:
        unique_labels.update(label_list)
    #print(unique_labels)
    df_1_label_dict = {label: [] for label in unique_labels}
    for index, row in df_1.iterrows():
        for label in row[label_column]:
            df_1_label_dict[label].append(row[uri_column])
    
    if '' in df_1_label_dict:
        del df_1_label_dict['']
    
    for _, row in tqdm(df_2.iterrows(), total=len(df_2), desc="match labels"):
        labels = row[label_column]
        uri_2 = row[uri_column]
        for label in labels:
            if label in df_1_label_dict:
                for uri_1 in df_1_label_dict[label]:
                    if uri_1 in prob_dict and uri_2 in prob_dict[uri_1]:
                        continue
                    prob_dict.setdefault(uri_1, {})[uri_2] = max_score

In [9]:
def normalize_string(s):
    # remove parenthesis
    s = re.sub("[\(\[].*?[\)\]]", "", s)
    # remove punctuation
    s = s.translate(str.maketrans('', '', string.punctuation))
    # split the string at uppercase letters and digits and join with spaces
    s = re.sub(r'([a-z])([A-Z])', r'\1 \2', s)
    s = re.sub(r'([A-Z])([A-Z][a-z])', r'\1 \2', s)
    s = re.sub(r'([a-zA-Z])(\d)', r'\1 \2', s)
    s = re.sub(r'(\d)([a-zA-Z])', r'\1 \2', s)
    # replace underscores with spaces
    s = s.replace('_', ' ')
    # remove multiple consecutive spaces and leading/trailing spaces
    s = ' '.join(s.split()).strip()
    # convert to lowercase and return
    return s.lower()

In [10]:
nltk.download('stopwords') # download stop words list

stop_words = set(stopwords.words('english')) # set of English stop words

def remove_stopwords(s):
    words = s.split() # split text into individual words

    filtered_words = [word for word in words if not word.lower() in stop_words] # remove stop words

    filtered_text = ' '.join(filtered_words)
    return filtered_text

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ch.Raghava\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
def remove_corpus_specific_stopword(graph_df):
    doc_freq = defaultdict(int)

    for _, row in graph_df.iterrows():
        doc = ' '.join(row['stopword label'])
        words = set(word_tokenize(doc))
        for word in words:
            doc_freq[word] += 1
            
    total_docs = len(graph_df)
    corpus_stopwords = set()
    for word, freq in doc_freq.items():
        if freq / total_docs > 0.2: # word appears in more than 20% of documents
            corpus_stopwords.add(word)    
    label_list = list()
    for _, row in graph_df.iterrows():
        filtered_text_list = list()
        for label in row['stopword label']:
            words = word_tokenize(label)
            filtered_words = [word for word in words if not word.lower() in corpus_stopwords]
            filtered_text = ' '.join(filtered_words)
            filtered_text_list.append(filtered_text)
        label_list.append(filtered_text_list)
    
    graph_df['corpus stopword label'] = label_list

In [12]:
nltk.download('punkt_tab')

nltk_tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Ch.Raghava\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [13]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained("bert-base-uncased").to(device)

#model_dir = '/Users/yamamotobikutorueiichi/codes/models/stella_en_1.5B_v5'
#model_dir = "dunzhang/stella_en_1.5B_v5"
#model = AutoModel.from_pretrained(model_dir).to(device)
#tokenizer = AutoTokenizer.from_pretrained(model_dir)

Using device: cpu


In [14]:
def embed_long_sentence(long_sentence):
    sentences = nltk_tokenizer.tokenize(long_sentence)
    
    tokens = {'input_ids': [], 'attention_mask': []}

    max_token = 140
    #for sentence in sentences:
    #    new_tokens = tokenizer.encode_plus(sentence, return_tensors="pt", max_length=max_token, padding='max_length')
    #    tokens['input_ids'].append(new_tokens['input_ids'][0][:max_token])
    #    tokens['attention_mask'].append(new_tokens['attention_mask'][0][:max_token])

    #tokens['input_ids'] = torch.stack(tokens['input_ids'])
    #tokens['attention_mask'] = torch.stack(tokens['attention_mask'])

    # Process sentences
    for sentence in sentences:
        new_tokens = tokenizer.encode_plus(sentence, return_tensors="pt", max_length=max_token, padding='max_length')
        tokens['input_ids'].append(new_tokens['input_ids'][0][:max_token])
        tokens['attention_mask'].append(new_tokens['attention_mask'][0][:max_token])
    
    # Stack and move tokens to the MPS device
    tokens['input_ids'] = torch.stack(tokens['input_ids']).to(device)
    tokens['attention_mask'] = torch.stack(tokens['attention_mask']).to(device)

    
    outputs = model(**tokens)
    
    embeddings = outputs.last_hidden_state
    
    attention_mask = tokens['attention_mask']
    
    mask = attention_mask.unsqueeze(-1).expand(embeddings.size()).float()

    masked_embeddings = embeddings * mask
    
    summed = torch.sum(masked_embeddings, 1)

    summed_mask = torch.clamp(mask.sum(1), min=1e-9)

    mean_pooled = summed / summed_mask
    
    return mean_pooled.detach().cpu().numpy()

In [15]:
def get_label_vec_dict(df):
    LABEL = 'corpus stopword label'

    #df_label_set = set(df[LABEL])
    unique_labels = set().union(*df[LABEL])

    df_label_vec_dict = dict()

    for label in tqdm(unique_labels):
        if not label:
            continue

        df_label_vec_dict[label] = embed_long_sentence(label)
    return df_label_vec_dict

In [16]:
def apply_match_label(prob_dict, df_1, df_2, uri_column, label_to_apply, new_label, max_score):
    df_1[new_label] = df_1[label_to_apply].apply(lambda label_list: [normalize_string(label) for label in label_list])

    df_2[new_label] = df_2[label_to_apply].apply(lambda label_list: [normalize_string(label) for label in label_list])
    
    match_labels(prob_dict, df_1, df_2, new_label, max_score, uri_column)

In [17]:
def calculate_prob_dict(df_1, df_2, uri_column):
    prob_dict = calculate_similarity_using_label(df_1, df_2, uri_column)
    
    match_labels(prob_dict, df_1, df_2, 'label', 0.9, uri_column)

    apply_match_label(prob_dict, df_1, df_2, uri_column, 'label', 'normalized label', 0.8)
    apply_match_label(prob_dict, df_1, df_2, uri_column, 'normalized label', 'stopword label', 0.7)

    remove_corpus_specific_stopword(df_1)
    remove_corpus_specific_stopword(df_2)
    
    # Fuzzy and BERT match only for list with less than MAX_LENGTH_FULL_MATCH
    if len(df_1) > MAX_LENGTH_FULL_MATCH or len(df_2) > MAX_LENGTH_FULL_MATCH:
        return prob_dict

    # Match using fuzzy string
    LABEL = 'corpus stopword label'
    for _, row in tqdm(df_1.iterrows(), total=len(df_1), desc='Corpus stopword match'):
        labels_1 = row[LABEL]
        uri_1 = row[uri_column]
        if not labels_1:
            continue

        for _, row_2 in df_2.iterrows():
            labels_2 = row_2[LABEL]
            uri_2 = row_2[uri_column]
            if not labels_2:
                continue
            if not uri_1 in prob_dict.keys() or not uri_2 in prob_dict[uri_1]:
                for label_1 in labels_1:
                    max_sim = 0.0
                    for label_2 in labels_2:
                        sim = fuzz.WRatio(label_1, label_2) / 100
                        prob = 0.7 * sim
                        if prob > max_sim:
                            max_sim = prob
                    prob_dict.setdefault(uri_1, {})[uri_2] = max_sim           
    
    # Match using BERT
    graph_1_label_vec_dict = get_label_vec_dict(df_1)
    graph_2_label_vec_dict = get_label_vec_dict(df_2)
    
    LABEL = 'corpus stopword label'
    for _, row in tqdm(df_2.iterrows(), total=len(df_2)):
        if uri_1 in prob_dict and uri_2 in prob_dict[uri_1] and prob_dict[uri_1][uri_2] > 0.7:
            continue
                
        labels_2 = row[LABEL]
        uri_2 = row[uri_column]
        if not labels_2:
            continue
        

        for _, row_1 in df_1.iterrows():
            uri_1 = row_1[uri_column]
            labels_1 = row_1[LABEL]

            if not labels_1:
                continue
            
            max_sim = 0.0
            for label_1 in labels_1:
                if not label_1:
                    continue
                for label_2 in labels_2:
                    if not label_2:
                        continue
                    vec_1 = graph_1_label_vec_dict[label_1]
                    vec_2 = graph_2_label_vec_dict[label_2]

                    similarities = cosine_similarity(vec_1, vec_2)
                    max_similarity = max(map(max, similarities))
                    prob = 0.7 * max_similarity
                    if prob > max_sim:
                        max_sim = prob

            if max_sim > prob_dict[uri_1][uri_2]:
                prob_dict[uri_1][uri_2] = max_sim
    
    return prob_dict

In [18]:
fname = FILE_FOLDER + 'predicate_sem_prob.csv'
predicate_prob_dict = dict()

if os.path.isfile(fname):
    predicate_sem_prob_df = pd.read_csv(fname)
    for _, row in tqdm(predicate_sem_prob_df.iterrows(), total=len(predicate_sem_prob_df)):
        p1 = row['p1']
        p2 = row['p2']
        sim = row['sim']
        predicate_prob_dict.setdefault(p1, dict())[p2] = sim
else:
    predicate_prob_dict = calculate_prob_dict(graph_1_inv_func_df, graph_2_inv_func_df, 'predicate')
    predicate_sem_prob_list = list()

    for pred_1, pred_2_dict in predicate_prob_dict.items():
        for pred_2 in pred_2_dict:
            predicate_sem_prob_list.append({
                'p1': pred_1,
                'p2': pred_2,
                'sim': pred_2_dict[pred_2]
            })
    predicate_sem_prob_df = pd.DataFrame(predicate_sem_prob_list)
    predicate_sem_prob_df.to_csv(fname)

  0%|          | 0/383135 [00:00<?, ?it/s]

In [19]:
threshold_graph_1_inv_func_df = graph_1_inv_func_df[(graph_1_inv_func_df[FUNCTIONALITY] > 0.25) | (graph_1_inv_func_df[INVERSE_FUNCTIONALITY] > 0.25)]
threshold_graph_2_inv_func_df = graph_2_inv_func_df[(graph_2_inv_func_df[FUNCTIONALITY] > 0.25) | (graph_2_inv_func_df[INVERSE_FUNCTIONALITY] > 0.25)]

#threshold_graph_1_inv_func_df = graph_1_inv_func_df
#threshold_graph_2_inv_func_df = graph_2_inv_func_df


In [20]:
def get_graph_classes(graph):
    query = """
        select * {?s a owl:Class.}
        """
    class_list = list()
    
    for res in graph.query(query):
        class_list.append(res[0])
    return class_list

In [21]:
def get_graph_entities(graph):
    #query = """
    #    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    #    SELECT DISTINCT ?s
    #    WHERE {
    #      ?s a ?type.
    #      FILTER NOT EXISTS {
    #        ?type rdf:type skos:Concept.
    #      }
    #    }"""
    query = """
    SELECT DISTINCT ?s
        WHERE {
          ?s <http://www.w3.org/2000/01/rdf-schema#label> ?o.
        }
    """
    entity_list = list()
    
    for res in graph.query(query):
        entity_list.append(str(res[0]))
    return entity_list

In [22]:
fname = FILE_FOLDER + 'entity_sem_prob.csv'
entity_sem_prob_dict = dict()

if os.path.isfile(fname):
    entity_sem_prob_df = pd.read_csv(fname)
    for _, row in entity_sem_prob_df.iterrows():
        e1 = row['e1']
        e2 = row['e2']
        sim = row['sim']
        entity_sem_prob_dict.setdefault(e1, dict())[e2] = sim
else:
    graph_1_entity_list = get_graph_entities(graph_1)
    graph_2_entity_list = get_graph_entities(graph_2)
    
    entity_1_df = pd.DataFrame()
    entity_1_df['entity'] = graph_1_entity_list
    entity_2_df = pd.DataFrame()
    entity_2_df['entity'] = graph_2_entity_list
    
    entity_sem_prob_dict = calculate_prob_dict(entity_1_df, entity_2_df, 'entity')

    entity_sem_prob_list = list()

    for entity_1 in entity_sem_prob_dict.keys():
        for entity_2 in entity_sem_prob_dict[entity_1].keys():
            entity_sem_prob_list.append({
                'e1': entity_1,
                'e2': entity_2,
                'sim': entity_sem_prob_dict[entity_1][entity_2]
            })
    entity_sem_prob_df = pd.DataFrame(entity_sem_prob_list)
    entity_sem_prob_df.to_csv(fname)

In [23]:
#threshold_graph_1_inv_func_df = threshold_graph_1_inv_func_df[threshold_graph_1_inv_func_df['predicate'].str.contains(LABEL_PREDICATE) == False]
#threshold_graph_2_inv_func_df = threshold_graph_2_inv_func_df[threshold_graph_2_inv_func_df['predicate'].str.contains(LABEL_PREDICATE) == False]



In [24]:
literal_y1_dict = dict()
used_p1_dict = dict()
graph_1_predicate_list = list()

for index, row in tqdm(threshold_graph_1_inv_func_df.iterrows(), total=len(threshold_graph_1_inv_func_df)):
    predicate = row[PREDICATE]
    predicate_uri = URIRef(predicate)

    if sum(1 for _ in graph_1.subject_objects(predicate_uri)) < 5:
        continue
    
    graph_1_predicate_list.append(predicate)
    
    #for s, p, o in graph_1.triples((None, predicate_uri, None)):
    #    if type(o) != Literal or not isinstance(o, Literal) or not (o.datatype is None or o.datatype == XSD.string or o.datatype == RDF.langString):
    #        continue
    for s, p, o in graph_1.triples((None, predicate_uri, None)):

        #if type(o) != Literal or not isinstance(o, Literal) or not (o.datatype is None or o.datatype == XSD.string or o.datatype == RDF.langString):
        #    continue
        
        #if type(o) != Literal or not isinstance(o, Literal):
        #    continue
        if type(o) != Literal:
            continue
        
        if (o.datatype is None or o.datatype == XSD.string or o.datatype == RDF.langString):
            o_value = str(o)
        else:
            o_value = o.toPython()
        #o_value = str(o)
        if o_value not in literal_y1_dict.keys():
            literal_y1_dict[o_value] = list()
        y1_value_list = literal_y1_dict[o_value]
        y1_value_list.append({
            SUBJECT: str(s),
            PREDICATE: str(predicate_uri)
        })

  0%|          | 0/545 [00:00<?, ?it/s]

In [25]:
def secure_triple_query(graph, s, p, o):
    triples = list()
    attempts = 0
    while attempts < 10:
        try:
            for s, p, o in graph.triples((s, p, o)):
                triples.append((s, p, o))
            return triples
        except Exception as e:
            print(e)
            attempts += 1
    return list()

In [26]:
loaded_attributes_dict = dict()

In [27]:
y2_matches = dict()
graph_2_predicate_list = list()

In [28]:
fname_matches = FILE_FOLDER + 'y2_matches.pkl'
fname_graph_2_predicate_list = FILE_FOLDER + 'graph_2_predicate_list.pkl'

if os.path.isfile(fname_matches) and os.path.isfile(fname_graph_2_predicate_list):
    with open(fname_matches, 'rb') as f:
        y2_matches = pickle.load(f)
    with open(fname_graph_2_predicate_list, 'rb') as f:
        graph_2_predicate_list = pickle.load(f)
    print('loaded y2_matches and graph_2_predicate_list')
else:
    for index, row in tqdm(threshold_graph_2_inv_func_df.iterrows(), total=len(threshold_graph_2_inv_func_df)):
        predicate = row[PREDICATE]
        predicate_uri = URIRef(predicate)
        
        #if sum(1 for _ in graph_2.subject_objects(predicate_uri)) < 5:
        #    continue
            
        if predicate in graph_2_predicate_list:
            continue

        graph_2_predicate_list.append(predicate)
        
        for s, p, o in secure_triple_query(graph_2, None, predicate_uri, None):
            #if type(o) != Literal or not isinstance(o, Literal) or not (o.datatype is None or o.datatype == XSD.string or o.datatype == RDF.langString):
            #    continue

            subject_triples = loaded_attributes_dict.setdefault(str(s), list())
            subject_triples.append({
                's': str(s),
                'p': str(p),
                'o': o
            })
            
            #if type(o) != Literal or not isinstance(o, Literal):
            #    continue
            if type(o) != Literal:
                continue
            # Check if exist match
            #o_value = str(o)
            if (o.datatype is None or o.datatype == XSD.string or o.datatype == RDF.langString):
                o_value = str(o)
            else:
                o_value = o.toPython()
            if o_value not in literal_y1_dict.keys():
                continue
                
            if o_value not in y2_matches.keys():
                y2_matches[o_value] = list()
            y2_match_list = y2_matches[o_value]
            y2_match_list.append({
                SUBJECT: str(s),
                PREDICATE: str(predicate_uri)
            })
    with open(FILE_FOLDER + 'y2_matches.pkl', 'wb') as f:  # open a text file
        pickle.dump(y2_matches, f) # serialize the list
    
    with open(FILE_FOLDER + 'graph_2_predicate_list.pkl', 'wb') as f:  # open a text file
        pickle.dump(graph_2_predicate_list, f) # serialize the list

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#gYear, Converter=<function parse_date at 0x000002868FE7A0D0>
Traceback (most recent call last):
  File "C:\Users\Ch.Raghava\miniconda3\envs\full-triple-matcher\lib\site-packages\rdflib\term.py", line 2119, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "C:\Users\Ch.Raghava\miniconda3\envs\full-triple-matcher\lib\site-packages\isodate\isodates.py", line 203, in parse_date
    raise ISO8601Error('Unrecognised ISO 8601 date format: %r' % datestring)
isodate.isoerror.ISO8601Error: Unrecognised ISO 8601 date format: '-0445'
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#gYear, Converter=<function parse_date at 0x000002868FE7A0D0>
Traceback (most recent call last):
  File "C:\Users\Ch.Raghava\miniconda3\envs\full-triple-matcher\lib\site-packages\rdflib\term.py", line 2119, in _castLexicalToPython
    return conv_func(le

loaded y2_matches and graph_2_predicate_list


In [29]:
func_1_dict = dict()

for index, row in tqdm(threshold_graph_1_inv_func_df.iterrows(), total=len(threshold_graph_1_inv_func_df)):
    func_1_dict[row['predicate']] = row[FUNCTIONALITY]

  0%|          | 0/545 [00:00<?, ?it/s]

In [30]:
inv_func_1_dict = dict()

for index, row in tqdm(threshold_graph_1_inv_func_df.iterrows(), total=len(threshold_graph_1_inv_func_df)):
    inv_func_1_dict[row['predicate']] = row[INVERSE_FUNCTIONALITY]

  0%|          | 0/545 [00:00<?, ?it/s]

In [31]:
func_2_dict = dict()

for index, row in tqdm(threshold_graph_2_inv_func_df.iterrows(), total=len(threshold_graph_2_inv_func_df)):
    func_2_dict[row['predicate']] = row[FUNCTIONALITY]

  0%|          | 0/703 [00:00<?, ?it/s]

In [32]:
inv_func_2_dict = dict()

for index, row in tqdm(threshold_graph_2_inv_func_df.iterrows(), total=len(threshold_graph_2_inv_func_df)):
    inv_func_2_dict[row['predicate']] = row[INVERSE_FUNCTIONALITY]

  0%|          | 0/703 [00:00<?, ?it/s]

In [33]:
high_sim_predicate_pair_dict = dict()

for index, row in tqdm(graph_1_inv_func_df.iterrows(), total=len(graph_1_inv_func_df)):
    p1 = row[PREDICATE]
    
    p1_functionality = func_1_dict[p1]
    
    if p1_functionality < 0.5:
        continue
    
    if p1 not in predicate_prob_dict:
        continue
    
    for p2 in predicate_prob_dict[p1]:
        p2_functionality = func_2_dict[p2]
        
        predicate_sim = predicate_prob_dict[p1][p2]
        
        func_sim_value = p1_functionality * p2_functionality * predicate_sim
        
        if func_sim_value > 0.5:
            high_sim_predicate_pair_dict.setdefault(p1, {})[p2] = func_sim_value

  0%|          | 0/545 [00:00<?, ?it/s]

In [34]:
import csv

In [35]:
def save_to_file():
    global global_saved_triple_list
    global global_iteration
    
    if len(global_saved_triple_list) == 0:
        return
    fieldnames = global_saved_triple_list[0].keys()
    
    # Save to CSV file
    with open(FILE_FOLDER + "saved_triples_" + str(global_iteration) + ".csv", 'a', newline='') as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames, delimiter='|')

        # Write data
        for row in global_saved_triple_list:
            try:
                writer.writerow(row)
            except Exception as e:
                print(e)
            
    global_saved_triple_list = list()

In [36]:
def save_triples(x1, p1, y1, x2, p2, y2, sim):
    if sim < 0.25:
        return
    global global_saved_triple_list
    global_saved_triple_list.append({'x1': x1,
                             'p1': p1,
                             'y1': y1,
                             'x2': x2,
                             'p2': p2,
                             'y2': y2,
                             'sim': sim})
    if len(global_saved_triple_list) > 10000:
        save_to_file()

In [37]:
def save_div_triples(div_triples, x1, p1, y1, x2, p2, y2, triple_sim, save=False):
    global global_iteration
    if save or len(div_triples) > 10000:
        with open(FILE_FOLDER + "div_triples_" + str(global_iteration) + ".csv", 'a', newline='') as csv_file:
            fieldnames = div_triples[0].keys()
            writer = csv.DictWriter(csv_file, fieldnames=fieldnames)

            # Write data
            for row in global_saved_triple_list:
                writer.writerow(row)
        return
    if triple_sim < 0.25:
        return
    div_triples.append({'x1': x1,
                         'p1': p1,
                         'y1': y1,
                         'x2': x2,
                         'p2': p2,
                         'y2': y2,
                         'sim': triple_sim})

In [38]:
def update_prob_entities(entity_pair_match_dict, x1, x2, p1, p2, prob_x, prob_y):
    y1_pred = str(p1)
    y2_pred = str(p2)
    if y1_pred not in predicate_prob_dict or y2_pred not in predicate_prob_dict[y1_pred]:
        return 0.0, 0.0
    pred_sim = predicate_prob_dict[y1_pred][y2_pred]
    
    prob_func = prob_x * pred_sim * func_1_dict[y1_pred] * func_2_dict[y2_pred] * prob_y
    prob_inv_func = prob_x * pred_sim * inv_func_1_dict[y1_pred] * inv_func_2_dict[y2_pred] * prob_y
    new_factor = (1-prob_func) * (1-prob_inv_func)
    
    if new_factor > 0.99:
        return 0.0, 0.0
    
    if x1 not in entity_pair_match_dict.keys():
        entity_pair_match_dict[x1] = dict()
    if x2 not in entity_pair_match_dict[x1].keys():
        entity_pair_match_dict[x1][x2] = 1

    entity_pair_match_dict[x1][x2] *= (1-prob_func) * (1-prob_inv_func)
    
    prob_triple = prob_func
    if prob_inv_func > prob_func:
        prob_triple = prob_inv_func

    return 1 - entity_pair_match_dict[x1][x2], 1 - new_factor

In [39]:
def calculate_entity_prob_using_attribute(entity_triple_match, entity_pair_dict):
    for o_value, y2_match_list in tqdm(y2_matches.items(), desc='Attribute loop', leave=False):
        y1_match_list = literal_y1_dict[o_value]

        min_size_for_tqdm = 100  # Define your threshold here

        if len(y1_match_list) > min_size_for_tqdm:
            iterable = tqdm(y1_match_list, desc='y1_match_list', leave=False)
        else:
            iterable = y1_match_list
        for y1_match in iterable:
            y1_subject = str(y1_match['subject'])

            for y2_match in y2_match_list:
                y2_subject = str(y2_match['subject'])

                #if y1_subject in entity_pair_dict and y2_subject in entity_pair_dict[y1_subject]:
                y1_pred = str(y1_match['predicate'])
                y2_pred = str(y2_match['predicate'])
                
                prob_x = 0.35
                if y1_subject in entity_pair_dict and y2_subject in entity_pair_dict[y1_subject]:
                    prob_x = entity_pair_dict[y1_subject][y2_subject]
                
                # Here we only update subject, because objects are literals
                sim, triple_sim = update_prob_entities(entity_triple_match, str(y1_subject), str(y2_subject),
                                     y1_pred, y2_pred, prob_x, 1)
                
                save_triples(y1_subject, y1_pred, o_value, y2_subject, y2_pred, o_value, triple_sim)

In [40]:
def get_neighbors(graph, predicate_list, y, loaded_neighbors_dict, inbound=True):
    if not y:
        return []
    
    if y in loaded_neighbors_dict.keys():
        return loaded_neighbors_dict[y]
    
    y_url = y
    if type(y_url) != URIRef:
        y_url = URIRef(y)
    
    attempts = 0
    while attempts < 10:
        try:
            neighbor_list = list()
            if inbound:
                for s, p in graph.subject_predicates(y_url):
                    if str(p) in predicate_list:
                        neighbor_list.append({
                            'p': str(p),
                            's': str(s)
                        })
            else:
                for p, o in graph.predicate_objects(y_url):
                    if str(p) in predicate_list:
                        neighbor_list.append({
                            'p': str(p),
                            'o': o
                        })
            return neighbor_list
        except Exception as e:
            print(e)
            attempts += 1
    loaded_neighbors_dict[y] = neighbor_list
    return neighbor_list

In [41]:
def get_entity_pair_sim_with_default(entity_pair_dict, e1, e2):
    if e1 in entity_pair_dict and e2 in entity_pair_dict[e1]:
        return entity_pair_dict[e1][e2]
    return 0.35

In [42]:
def save_triples_with_both_side_as_entities(entity_triple_match, entity_pair_dict, s1, s2, p1, p2, o1, o2):
    s_sim = get_entity_pair_sim_with_default(entity_pair_dict, s1, s2)
    o_sim = get_entity_pair_sim_with_default(entity_pair_dict, o1, o2)
    sim, triple_sim = update_prob_entities(entity_triple_match, s1, s2, p1, p2, s_sim, o_sim)
    sim, triple_sim = update_prob_entities(entity_triple_match, o1, o2, p1, p2, s_sim, o_sim)
    save_triples(s1, p1, o1, s2, p2, o2, triple_sim)

In [43]:
def update_prob_entity_neighbors(entity_triple_match, entity_pair_dict, y1_neighbors,
                                 y2_neighbors, y_sim, y1, y2):
    for x1 in y1_neighbors:
        for x2 in y2_neighbors:
            x1_str = str(x1['s'])
            x2_str = str(x2['s'])
            p1_str = str(x1['p'])
            p2_str = str(x2['p'])
            x_sim = 0.35
            if x1_str in entity_pair_dict and x2_str in entity_pair_dict[x1_str]:
                x_sim = entity_pair_dict[x1_str][x2_str]
            
            save_triples_with_both_side_as_entities(entity_triple_match, entity_pair_dict, x1_str, x2_str,
                                                   p1_str, p2_str, y1, y2)

In [44]:
def calculate_entity_prob_using_neighbors(entity_triple_match, entity_pair_dict, loaded_neighbors_dict):
    y1_keys = list(entity_pair_dict.keys())
    predicate_1_list = list(graph_1_inv_func_df[graph_1_inv_func_df[INVERSE_FUNCTIONALITY] > 0.25][PREDICATE])
    predicate_2_list = list(graph_2_inv_func_df[graph_2_inv_func_df[INVERSE_FUNCTIONALITY] > 0.25][PREDICATE])

    for y1 in tqdm(y1_keys, desc='neighbor loop', leave=False):
        y1_neighbors = get_neighbors(graph_1, predicate_1_list, y1, loaded_neighbors_dict, True)
        
        entity_2_top_n_list = get_top_n(entity_pair_dict[y1], 10)

        for y2, y_sim in entity_2_top_n_list:
            #if y_sim < 0.25:
            #    continue
            
            y2_neighbors = get_neighbors(graph_2, predicate_2_list, y2, loaded_neighbors_dict, True)
            update_prob_entity_neighbors(entity_triple_match, entity_pair_dict,
                                         y1_neighbors, y2_neighbors, y_sim, y1, y2)

In [45]:
def match_attributes(entity_triple_match, entity_pair_dict, e1_attributes,
                    e2_attributes, e_sim, e1, e2, div_triples):
    for e1_attribute in e1_attributes:
        p1_str = str(e1_attribute['p'])
        o1 = e1_attribute['o']
        for e2_attribute in e2_attributes:
            p2_str = str(e2_attribute['p'])
            o2 = e2_attribute['o']
            o_sim = 0.0
            if type(o1) == URIRef and type(o2) == URIRef:
                o1_str = str(o1)
                o2_str = str(o2)
                if o1_str in entity_pair_dict and o2_str in entity_pair_dict[o1_str]:
                    o_sim = entity_pair_dict[o1_str][o2_str]
                else:
                    o_sim = 0.35
                save_triples_with_both_side_as_entities(entity_triple_match, entity_pair_dict,
                                                       e1, e2, p1_str, p2_str, o1_str, o2_str)
            else:
                o_sim = graph_similarity.get_objects_similarity(o1, o2)

                sim, triple_sim = update_prob_entities(entity_triple_match, e1, e2, p1_str, p2_str, e_sim, o_sim)
                save_triples(e1, p1_str, o1, e2, p2_str, o2, triple_sim)

            #div_sim = 0.0
            #if o_sim != 0:
            #    div_sim = triple_sim / o_sim * o_div
            #save_div_triples(div_triples, e1, p1_str, o1, e2, p2_str, o2, div_sim)

In [46]:
def calculate_entity_prob_using_all_attributes(entity_triple_match, entity_pair_dict, loaded_attributes_dict):
    e1_keys = list(entity_pair_dict.keys())
    div_triples = list()
    predicate_1_list = list(graph_1_inv_func_df[ (graph_1_inv_func_df[FUNCTIONALITY] > 0.8) |
                                                (graph_1_inv_func_df[INVERSE_FUNCTIONALITY] > 0.8)][PREDICATE])
    predicate_2_list = list(graph_2_inv_func_df[ (graph_2_inv_func_df[FUNCTIONALITY] > 0.8) |
                                                (graph_2_inv_func_df[INVERSE_FUNCTIONALITY] > 0.8)][PREDICATE])
    for e1 in tqdm(e1_keys, desc='All attributes loop', leave=False):
        e1_attributes = get_neighbors(graph_1, predicate_1_list, e1, loaded_attributes_dict, False)
        
        entity_2_top_n_list = get_top_n(entity_pair_dict[e1], 10)
        
        for e2, e_sim in entity_2_top_n_list:
            
            e2_attributes = get_neighbors(graph_2, predicate_2_list, e2, loaded_attributes_dict, False)
            match_attributes(entity_triple_match, entity_pair_dict, e1_attributes, e2_attributes, e_sim,
                             e1, e2, div_triples)
            
    #save_div_triples(div_triples, None, None, None, None, None, None, None, save=True)

In [47]:
def get_prob_entity_pair(entity_pair_match_dict, s1, s2):
    if s1 in entity_pair_match_dict.keys() and s2 in entity_pair_match_dict[s1].keys():
        return 1 - entity_pair_match_dict[s1][s2]
    return 0

In [48]:
def get_columns(reversed_bool):
    if reversed_bool:
        return {
            'x1': 'x2',
            'p1': 'p2',
            'y1': 'y2',
            'x2': 'x1',
            'p2': 'p1',
            'y2': 'y1'
        }
    else:
        return {
            'x1': 'x1',
            'p1': 'p1',
            'y1': 'y1',
            'x2': 'x2',
            'p2': 'p2',
            'y2': 'y2'
        }

In [49]:
def calculate_product(matched_row, entity_pair_match_dict):
    x_sim = 1 - entity_pair_match_dict[matched_row['x1']][matched_row['x2']]
    if matched_row['y1'] == matched_row['y2']:
        y_sim = 1
    else:
        y_sim = entity_pair_match_dict[matched_row['y1']][matched_row['y2']]
    return 1 - x_sim * y_sim

In [50]:
def calculate_sum_by_triple(indexed_df, triple_row):
    matched_triples = indexed_df.join(triple_row)
    return 1.0 - numpy.product(matched_triples['product_element'])

In [51]:
def calculate_x_sim(x1, x2, entity_pair_match_dict, reversed_bool):
    if reversed_bool:
        return 1 - entity_pair_match_dict[x2][x1]
    return 1 - entity_pair_match_dict[x1][x2]

In [52]:
def calculate_y_sim(y1, y2, entity_pair_match_dict, reversed_bool):
    if y1 == y2:
        return 1
    else:
        if reversed_bool:
            if y2 in entity_pair_match_dict.keys() and y1 in entity_pair_match_dict[y2].keys():
                return 1 - entity_pair_match_dict[y2][y1]
        if y1 in entity_pair_match_dict.keys() and y2 in entity_pair_match_dict[y1].keys():
            return 1 - entity_pair_match_dict[y1][y2]
    return 0

In [53]:
def get_subject_object(graph, predicate):
    attempts = 0
    while attempts < 10:
        try:
            s_o_list = list()
            for s, o in graph.subject_objects(URIRef(predicate)):
                s_o_list.append({
                        'subject': s,
                        'object': o
                    })
            return s_o_list
        except Exception as e:
            print(e)
            attempts += 1
    
    return s_o_list

In [54]:
def check_if_string(y):
    return type(y) == Literal and y.value and type(y.value) == str

In [55]:
def calculate_sub_pred_product(entity_pair_match_dict, x1, x2, y1, y2):
    if check_if_string(y1) and check_if_string(y2):
        if y1.value != y2.value:
            return 0.0
        y_value = y1.value
        if y_value not in literal_y1_dict.keys() or y_value not in y2_matches.keys():
            return 0.0
        return entity_pair_match_dict[str(x1)][str(x2)]
    return 0.0

In [56]:
def save_entity_pairs(i, entity_pair_dict):
    entity_pair_sim_list = list()
    for y1 in entity_pair_dict.keys():
        for y2 in entity_pair_dict[y1].keys():
            if entity_pair_dict[y1][y2] < 0.01:
                continue
            entity_pair_sim_list.append({
                "e1": y1,
                "e2": y2,
                "sim": entity_pair_dict[y1][y2]
            })
    entity_sim_df = pd.DataFrame(entity_pair_sim_list)
    entity_sim_df.to_csv(FILE_FOLDER + "entity_sim_" + str(i) + ".csv")

In [57]:
def get_entity_vec(graph, entity):
    if entity in label_dict:
        return label_dict[entity]
    predicate = URIRef(LABEL_PREDICATE)
    entity_uri = URIRef(entity)
    
    label = str(next(graph.objects(entity_uri, predicate), ""))
    
    if not label:
        return None
    
    vec = embed_long_sentence(label)
    label_dict[entity] = vec
    
    return vec

In [58]:
def entity_label_bert_similarity(graph_1, graph_2, entity_1, entity_2):
    vec_1 = get_entity_vec(graph_1, entity_1)
    vec_2 = get_entity_vec(graph_2, entity_2)
    
    if vec_1 is None or vec_2 is None:
        return 0.0
    
    similarities = cosine_similarity(vec_1, vec_2)
    return similarities.mean()

In [59]:
def pre_calc_entity_pair_dict(entity_sem_prob_dict, entity_triple_match):
    entity_prob_dict = dict()
    
    for entity_1, entity_2_dict in entity_sem_prob_dict.items():
        for entity_2 in entity_2_dict:
            entity_prob_dict.setdefault(entity_1, {})[entity_2] = 0.5 * entity_sem_prob_dict[entity_1][entity_2]
    
    for entity_1, entity_2_dict in tqdm(entity_triple_match.items(), desc='entity_pair', leave=False):
        entity_2_top_n_list = get_top_n(entity_2_dict, 10, False)
        #entity_2_top_n_list = get_top_n(entity_2_dict, 5, False)
        
        for entity_2, triple_match_value in entity_2_top_n_list:
            entity_triple_sim = 1 - triple_match_value
            
            if entity_2 not in entity_prob_dict.setdefault(entity_1, {}):
                #if entity_triple_sim < 0.2:
                #    entity_prob_dict.setdefault(entity_1, {})[entity_2] = entity_triple_sim
                #else:
                bert_sim = 0.8 * entity_label_bert_similarity(graph_1, graph_2, entity_1, entity_2)
                entity_sem_prob_dict.setdefault(entity_1, {})[entity_2] = bert_sim
                entity_prob_dict.setdefault(entity_1, {})[entity_2] = 0.5 * bert_sim
            entity_prob_dict.setdefault(entity_1, {})[entity_2] += 0.5 * entity_triple_sim
            
    return entity_prob_dict

In [60]:
def get_top_n(dictionary, n, reverse=True):
    sorted_elements = sorted(dictionary.items(), key=lambda x: x[1], reverse=reverse)
    if len(sorted_elements) < n:
        return sorted_elements
    n_sim = sorted_elements[n-1][1]
    
    top_n_list = list()
    for element in sorted_elements:
        e2, sim = element
        if (reverse and sim < n_sim) or (not reverse and sim > n_sim):
            return top_n_list[:20]
            
        top_n_list.append(element)
    return top_n_list[:20]

In [61]:
def save_triple_pairs(i, entity_pair_dict):
    entity_pair_sim_list = list()
    for y1 in entity_pair_dict.keys():
        for y2 in entity_pair_dict[y1].keys():
            if entity_pair_dict[y1][y2] < 0.01:
                continue
            entity_pair_sim_list.append({
                "e1": y1,
                "e2": y2,
                "sim": entity_pair_dict[y1][y2]
            })
    entity_sim_df = pd.DataFrame(entity_pair_sim_list)
    entity_sim_df.to_csv(FILE_FOLDER + "triple_sim_" + str(i) + ".csv")

In [62]:
def convert_df_to_entity_dict(df):
    grouped_df = df.groupby('e1')
    elem_dict = dict()
    for e1, group_indices in tqdm(grouped_df.groups.items()):
        # Access the group corresponding to 'e1'
        e1_group = df.loc[group_indices]
        e1_dict = dict(zip(e1_group['e2'], e1_group['sim']))
        elem_dict[e1] = e1_dict
    return elem_dict

In [ ]:
loaded_neighbors_dict = dict()
pred_sub_relation_1 = dict()
pred_sub_relation_2 = dict()
entity_triple_match = dict()
label_dict = dict()
entity_pair_dict = pre_calc_entity_pair_dict(entity_sem_prob_dict, entity_triple_match)
global_iteration = 0
global_saved_triple_list = list()
loaded_attributes_dict = dict()

for i in tqdm(range(10), desc='Main loop'):
    global_iteration = i
    
    calculate_entity_prob_using_attribute(entity_triple_match, entity_pair_dict)     
    
    calculate_entity_prob_using_neighbors(entity_triple_match, entity_pair_dict, loaded_neighbors_dict)
    calculate_entity_prob_using_all_attributes(entity_triple_match, entity_pair_dict, loaded_attributes_dict)

    save_triple_pairs(i, entity_triple_match)
    entity_pair_dict = pre_calc_entity_pair_dict(entity_sem_prob_dict, entity_triple_match)
    entity_triple_match = dict()
    save_to_file()

    if i > 0:
        previous_entity_df = pd.read_csv(FILE_FOLDER + "entity_sim_" + str(i-1) + ".csv")
        previous_entity_dict = convert_df_to_entity_dict(previous_entity_df)
        if len(entity_pair_dict.keys()) > 1.1 * len(previous_entity_dict.keys()):
            save_entity_pairs(i, entity_pair_dict)
            continue
        changed_top_1_counter = 0
        for e1, e1_dict in entity_pair_dict.items():
            if e1 not in entity_pair_dict.keys():
                changed_top_1_counter += 1
                continue
            previous_top_1 = get_top_n(e1_dict, 1)[0]
            current_top_1 = get_top_n(entity_pair_dict[e1], 1)[0]
            if previous_top_1 != current_top_1:
                changed_top_1_counter += 1
        if changed_top_1_counter < (0.1 * len(previous_entity_dict.keys())):
            break
    save_entity_pairs(i, entity_pair_dict)

entity_pair: 0it [00:00, ?it/s]

Main loop:   0%|          | 0/10 [00:00<?, ?it/s]

Attribute loop:   0%|          | 0/43939 [00:00<?, ?it/s]

y1_match_list:   0%|          | 0/188 [00:00<?, ?it/s]

y1_match_list:   0%|          | 0/201 [00:00<?, ?it/s]

...

C:\Users\Ch.Raghava\miniconda3\envs\full-triple-matcher\lib\site-packages\dateutil\parser\_parser.py:1207: UnknownTimezoneWarning: tzname UPA identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "


[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid a

C:\Users\Ch.Raghava\miniconda3\envs\full-triple-matcher\lib\site-packages\dateutil\parser\_parser.py:1207: UnknownTimezoneWarning: tzname JM identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "


[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid argument
[Errno 22] Invalid a

entity_pair:   0%|          | 0/71899 [00:00<?, ?it/s]